In [1]:
%cd /workspace/Code/experimental/

/workspace/Code/experimental


In [2]:
from data.datamodule import TextDataModule
from src.modules.core.encoder import TransformerEncoder
from src.modules.mlm_baseline.mlm_module import MLMModule
import lightning as L
import torch
from data.tokenizer import TokenizerWrapper
from lightning.pytorch.loggers import WandbLogger

In [ ]:
from src.modules.jepa.train_jepa import build_datamodule, build_module, build_callbacks, build_logger
import lightning as L
import argparse


In [ ]:
args = argparse.Namespace(
    # Données
    train_path   = "data/raw/tinystories_train.parquet",
    val_path     = "data/raw/tinystories_val.parquet",
    window_size  = 128,
    stride       = None,
    alpha        = 0.15,
    batch_size   = 32,
    num_workers  = 5,
    # Modèle
    backbone     = "bert-base-uncased",
    d_final      = 256,
    n_layers     = 3,
    n_heads      = 4,
    d_ff         = 512,
    dropout      = 0.1,
    # Optimisation
    lr           = 1e-4,
    weight_decay = 1e-2,
    max_epochs   = 20,
    patience     = 5,
    precision    = "16-mixed",
    # Infra
    output_dir   = "logs/jepa",
    run_name     = "jepa_tinystories_v1",
    logger       = "wandb",   # ou "wandb"
    wandb_project= "text-jepa",
    log_every    = 50,
    val_interval = 500,
    seed         = 42,
    resume       = None,
)


In [5]:
L.seed_everything(args.seed)
dm     = build_datamodule(args)
module = build_module(args)

Seed set to 42


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
total     = sum(p.numel() for p in module.parameters())
trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
print(f"Total      : {total:,}")
print(f"Entraînable: {trainable:,}  ({100*trainable/total:.1f}%)")

Total      : 33,361,664
Entraînable: 9,920,768  (29.7%)


In [ ]:
trainer = L.Trainer(
    max_epochs      = args.max_epochs,
    accelerator     = "auto",
    devices         = "auto",
    precision       = args.precision,
    logger          = build_logger(args),
    callbacks       = build_callbacks(args),
    gradient_clip_val = 1.0,
    log_every_n_steps = args.log_every,
    val_check_interval= args.val_interval,
)

trainer.fit(module, datamodule=dm)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Chargement du tokenizer depuis le cache local : ./data/tokenizers/bert-base-uncased


Loading `train_dataloader` to estimate number of stepping batches.
wandb: WARNING `wandb.require('service')` is a no-op as it is now the default behavior.
wandb: WARNING `wandb.require('service')` is a no-op as it is now the default behavior.
/home/marceau/.conda/envs/LE/lib/python3.14/multiprocessing/spawn.py:132: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  self = reduction.pickle.load(from_parent)
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
/home/marceau/.conda/envs/LE/lib/python3.14/multiprocessing/spawn.py:132: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  self = reduction.pickle.load(from_parent)
/home/marceau/.conda/envs/LE/lib/python3.14/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by t

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedder  │ FrozenBertEmbedding │ 23.4 M │ train │     0 │
│ 1 │ encoder   │ JEPAEncoder         │  9.7 M │ train │     0 │
│ 2 │ predictor │ JEPAPredictor       │  262 K │ train │     0 │
└───┴───────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 9.9 M                                                                                            
Non-trainable params: 23.4 M                                                                                       
Total params: 33.4 M                                                                                               
Total estimated model params size (MB): 133.447                                                                    
Modules in train mode: 45                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

wandb: WARNING `wandb.require('service')` is a no-op as it is now the default behavior.

/workspace/Code/experimental/data/masking.py:89: UserWarning: alpha=0.3 would mask 1/1 tokens — all tokens will be masked
  warnings.warn(


wandb: WARNING `wandb.require('service')` is a no-op as it is now the default behavior.

/home/marceau/.conda/envs/LE/lib/python3.14/multiprocessing/spawn.py:132: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  self = reduction.pickle.load(from_parent)
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.


/home/marceau/.conda/envs/LE/lib/python3.14/multiprocessing/spawn.py:132: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  self = reduction.pickle.load(from_parent)
/workspace/Code/experimental/data/masking.py:89: UserWarning: alpha=0.3 would mask 1/1 tokens — all tokens will be masked
  warnings.warn(


In [ ]:
print(f"Meilleur checkpoint : {trainer.checkpoint_callback.best_model_path}")